# NB17 MEV와 거래 순서 — 샌드위치 시뮬레이션(15장 풀 재사용)

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB17-1: 샌드위치 — 15장의 상수곱 풀을 그대로 가져와 「앞·피해자·뒤」 세 거래를 순서대로 넣는다 (공부용, 풀 단위)
import math

class ConstantProductPool:                                 # 15장 NB15-1 과 같은 클래스(스왑 부분만)
    def __init__(self, x, y, fee=0.003):
        self.x, self.y, self.fee = float(x), float(y), fee
    @property
    def price_x_in_y(self): return self.y / self.x
    def swap_x_for_y(self, dx):
        eff = dx * (1 - self.fee); dy = self.y * eff / (self.x + eff)
        self.x += dx; self.y -= dy; return dy
    def swap_y_for_x(self, dy):
        eff = dy * (1 - self.fee); dx = self.x * eff / (self.y + eff)
        self.y += dy; self.x -= dx; return dx

def sandwich(pool_x, pool_y, victim_y, attacker_y):
    """피해자는 Y victim_y 로 X 를 산다. 공격자는 그 앞에서 Y attacker_y 로 X 를 사고, 뒤에서 그 X 를 전부 되판다.
    반환: (공격자가 돌려받은 Y − 넣은 Y, 피해자가 받은 X, 피해자가 혼자였다면 받았을 X)"""
    alone = ConstantProductPool(pool_x, pool_y).swap_y_for_x(victim_y)
    p = ConstantProductPool(pool_x, pool_y)
    front_x = p.swap_y_for_x(attacker_y) if attacker_y > 0 else 0.0     # ① 앞: 공격자 매수
    victim_x = p.swap_y_for_x(victim_y)                                 # ② 피해자 매수 — 이미 오른 가격으로
    back_y = p.swap_x_for_y(front_x) if attacker_y > 0 else 0.0         # ③ 뒤: 공격자 매도
    return back_y - attacker_y, victim_x, alone

print("풀 X 1,000 / Y 1,000. 피해자는 Y 100 으로 X 를 산다.")
print(f"{'공격자 투입 Y':>12} {'공격자 순이익(Y)':>14} {'피해자가 받은 X':>14} {'혼자였다면 X':>12} {'피해자 손실 X':>12} {'피해자 손실 비율':>12}")
for a in [0, 50, 100, 200, 400, 800, 2000, 5000, 20000, 100000]:
    profit, vx, alone = sandwich(1000, 1000, 100, a)
    print(f"{a:>12} {profit:>14.3f} {vx:>14.3f} {alone:>12.3f} {alone - vx:>12.3f} {1 - vx / alone:>12.2%}")

풀 X 1,000 / Y 1,000. 피해자는 Y 100 으로 X 를 산다.
    공격자 투입 Y     공격자 순이익(Y)      피해자가 받은 X      혼자였다면 X     피해자 손실 X    피해자 손실 비율
           0          0.000         90.661       90.661        0.000        0.00%
          50          9.353         82.601       90.661        8.060        8.89%
         100         17.368         75.570       90.661       15.091       16.65%
         200         30.322         63.957       90.661       26.704       29.45%
         400         48.047         47.526       90.661       43.135       47.58%
         800         66.972         29.196       90.661       61.466       67.80%
        2000         84.967         10.743       90.661       79.918       88.15%
        5000         92.174          2.731       90.661       87.930       96.99%
       20000         94.020          0.226       90.661       90.435       99.75%
      100000         94.020          0.010       90.661       90.651       99.99%


In [ ]:
# NB17-2: 피해자의 슬리피지 허용치가 공격 크기를 묶는다 — 「최소 수취량」 조건을 넣고 공격자의 최적 투입을 찾는다 (공부용, 풀 단위)
def best_attack(pool_x, pool_y, victim_y, tolerance, fee=0.003, grid=2000):
    """피해자가 「혼자였다면 받을 X 의 (1 − tolerance) 이상은 받아야 한다」는 조건을 거래에 붙였다고 가정.
    그 조건을 깨지 않는(=피해자 거래가 되돌려지지 않는) 공격자 투입 Y 가운데 순이익이 가장 큰 것을 격자 탐색으로 찾는다."""
    alone = ConstantProductPool(pool_x, pool_y, fee).swap_y_for_x(victim_y)
    min_out = alone * (1 - tolerance)
    best = (0.0, 0.0)                                          # (순이익, 투입)
    for i in range(1, grid + 1):
        a = pool_y * 2 * i / grid                              # 0 ~ 풀 Y 의 두 배까지 훑는다
        profit, vx, _ = sandwich(pool_x, pool_y, victim_y, a)
        if vx < min_out:                                       # 피해자 거래가 실패하면 샌드위치도 성립하지 않는다(앞 거래만 남아 공격자 손해)
            break
        if profit > best[0]:
            best = (profit, a)
    return best, alone, min_out

print("풀 X 1,000 / Y 1,000. 피해자 주문 Y 100. 피해자가 거래에 붙인 슬리피지 허용치별로 공격자의 최적 투입과 순이익:")
print(f"{'허용치':>8} {'피해자 최소 수취 X':>16} {'공격자 최적 투입 Y':>16} {'공격자 순이익(Y)':>14}")
for tol in [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.50]:
    (profit, a), alone, min_out = best_attack(1000, 1000, 100, tol)
    print(f"{tol:>8.1%} {min_out:>16.3f} {a:>16.1f} {profit:>14.3f}")
print("\n허용치가 좁을수록 끼울 수 있는 공격이 작아진다(이 1 Y 격자에서는 0.1%일 때 0). 순이익은 허용치에 거의 비례하고, 최적 투입은 그보다 빠르게 커진다.")

풀 X 1,000 / Y 1,000. 피해자 주문 Y 100. 피해자가 거래에 붙인 슬리피지 허용치별로 공격자의 최적 투입과 순이익:
     허용치      피해자 최소 수취 X      공격자 최적 투입 Y     공격자 순이익(Y)
    0.1%           90.570              0.0          0.000
    0.5%           90.208              2.0          0.404
    1.0%           89.754              5.0          1.004
    2.0%           88.848             10.0          1.992
    5.0%           86.128             27.0          5.234
   10.0%           81.595             56.0         10.380
   50.0%           45.331            434.0         50.317

허용치가 좁을수록 끼울 수 있는 공격이 작아진다(이 1 Y 격자에서는 0.1%일 때 0). 순이익은 허용치에 거의 비례하고, 최적 투입은 그보다 빠르게 커진다.
